# Equipo 6 · Frecuencia, agregación y posible estacionalidad

**Curso:** Series temporales financieras
**Fuente de datos:** Yahoo Finance mediante `yfinance`
**Activo principal:** AAPL (Apple Inc.)
**Período de trabajo:** enero de 2020 a diciembre de 2025

## Misión del equipo
**Pregunta central:** ¿Qué cambia al resumir precios diarios en datos mensuales y qué evidencia sería necesaria para hablar de estacionalidad?

### Objetivos
- Distinguir frecuencia diaria y mensual.
- Reagrupar observaciones por mes.
- Diferenciar patrón estacional de una fluctuación casual.

> **Nota de reproducibilidad:** los resultados numéricos dependen de los datos que Yahoo Finance entregue al momento de ejecutar. Por eso las interpretaciones de este cuaderno se apoyan en celdas que calculan y muestran las cifras al ejecutarse, en lugar de cifras escritas a mano.


## 1. Investigación previa

**Definición de frecuencia.** 
Frecuencia es el intervalo con el que se registran u observan los datos de una serie temporal: diaria, mensual, trimestral o anual, por ejemplo. Indica cuántas observaciones hay por unidad de tiempo.

La frecuencia de una serie temporal es el intervalo regular con el que se registra una observación: diaria, semanal, mensual, trimestral, anual. Define la "resolución" de la serie y cuántas observaciones hay por unidad de tiempo (por ejemplo, 12 por año en una serie mensual). En una serie bursátil diaria la frecuencia no es perfectamente regular, porque solo hay datos en días hábiles de mercado (se omiten fines de semana y festivos), por lo que cada mes tiene entre ~19 y ~23 observaciones. 

**Remuestrear o reagrupar temporalmente (*resampling*).** Remuestrear es cambiar la frecuencia de una serie temporal. Al reagrupar hacia abajo (de mensual a trimestral), se combinan varios datos mediante una función como suma o promedio; al reagrupar hacia arriba (de mensual a diaria), se crean nuevos puntos que suelen requerir interpolación. (Banco de España Eurosistema, 2024).

**¿Qué hace la agregación mensual?** Agrupa todas las observaciones de un mes calendario y las reduce a un solo número. Aquí usamos el **último precio disponible del mes**, de modo que el rendimiento mensual es el cambio entre dos cierres consecutivos de mes. Es distinto de promediar los precios del mes, que suaviza la serie y puede distorsionar el rendimiento. Se gana una serie más limpia y comparable, y se pierde la dinámica intra-mes (volatilidad diaria, gaps, picos y caídas dentro del mes). (DANE, 2016).

**Estacionalidad vs. tendencia.**
- *Tendencia:* movimiento de largo plazo, sostenido, hacia arriba o hacia abajo, sin un calendario fijo.
- *Estacionalidad:* patrón que se repite con periodo fijo y conocido (cada 12 meses, cada semana, cada trimestre), ligado al calendario.
- *Componente irregular (residuo):* lo que queda cuando se quitan los dos anteriores.
Una serie puede crecer (tendencia) sin tener estacionalidad, y viceversa. Si el periodo del patrón no es fijo, se habla de comportamiento cíclico, no estacional. (Instituto Nacional de Estadística, s.f.)

**¿Qué evidencia permitiría sostener que un patrón estacional se repite?**
1. Que el patrón aparezca en **la mayoría de los años** de forma consistente (mismo signo y magnitud comparable), no solo en el promedio.
2. Que la diferencia entre meses sea **estadísticamente distinguible del azar**, usando una prueba formal y teniendo en cuenta que se comparan 12 grupos a la vez.
3. Que supere la **variabilidad natural** del activo: la dispersión dentro de cada mes debe ser pequeña frente a la diferencia entre meses.
4. Una **descomposición** (tendencia + estacional + residuo) donde el componente estacional tenga peso relevante.
5. **Validación fuera de muestra**: que el patrón detectado en unos años se mantenga en años posteriores no usados para encontrarlo.
6. Un **mecanismo económico plausible** que lo explique (por ejemplo, ciclos de producto, temporada de resultados, flujos de fin de año).


In [ ]:
!pip -q install yfinance

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TICKER = "AAPL"
START = "2020-01-01"
END = "2026-01-01"   # la fecha final de yfinance es exclusiva: el último dato es de diciembre de 2025

datos = yf.download(
    TICKER,
    start=START,
    end=END,
    auto_adjust=True,
    progress=False
)

# Algunas versiones de yfinance pueden devolver columnas MultiIndex.
if isinstance(datos.columns, pd.MultiIndex):
    datos.columns = datos.columns.get_level_values(0)

datos = datos.dropna(how="all")
datos.head()

### 1.1 Verificación de la frecuencia diaria
Antes de agregar, comprobamos qué frecuencia tienen realmente los datos: rango de fechas, número de observaciones y cuántos días de mercado hay por mes. Como `auto_adjust=True`, `Close` ya está ajustado por dividendos y splits, de modo que los rendimientos reflejan el retorno total aproximado.

In [ ]:
print(f"Primera fecha : {datos.index.min().date()}")
print(f"Última fecha  : {datos.index.max().date()}")
print(f"Observaciones : {len(datos)}")
print(f"Nulos en Close: {datos['Close'].isna().sum()}")

# Distancia entre observaciones consecutivas (en días calendario)
saltos = datos.index.to_series().diff().dt.days.value_counts().sort_index()
print("\nDías calendario entre observaciones consecutivas:")
print(saltos.to_string())

# Días de mercado por mes
dias_por_mes = datos["Close"].resample("ME").count()
print("\nDías de negociación por mes -> mínimo:", dias_por_mes.min(),
      "| máximo:", dias_por_mes.max(), "| promedio:", round(dias_por_mes.mean(), 1))

**Lectura:** los saltos de 1 día (lunes a viernes) y de 3 días (viernes a lunes) muestran que la serie es *diaria de días hábiles*, no diaria calendario. Los saltos mayores corresponden a festivos del mercado. Esto es importante: la frecuencia es "diaria" pero **no es perfectamente regular**, y los meses no tienen el mismo número de observaciones.

## 2. Agregar datos a frecuencia mensual
`resample('ME')` agrupa las observaciones por fin de mes. Calcularemos el último precio disponible de cada mes y el rendimiento entre cierres mensuales. Esto no equivale a promediar todos los precios del mes.

**Cómo se calcula (explicado con nuestras palabras):**
- `datos["Close"].resample("ME").last()` toma, para cada mes, el cierre del último día hábil.
- `.dropna()` elimina meses sin datos.
- `.pct_change() * 100` calcula $r_t = \frac{P_t - P_{t-1}}{P_{t-1}} \times 100$, el rendimiento porcentual entre dos cierres mensuales consecutivos. El primer mes queda como `NaN` porque no tiene un cierre previo dentro de la muestra.

In [ ]:
try:
    mensual = datos["Close"].resample("ME").last().dropna().to_frame("Cierre_mensual")
except ValueError:
    # pandas antiguos (< 2.2) usan "M" en lugar de "ME"
    mensual = datos["Close"].resample("M").last().dropna().to_frame("Cierre_mensual")

mensual["Rendimiento_mensual_pct"] = mensual["Cierre_mensual"].pct_change() * 100
display(mensual.head(12).round(2))

### 2.1 Verificación de los cálculos
Comprobamos a mano que el cierre mensual coincide con el último día hábil del mes y que el rendimiento se calculó correctamente, y cuantificamos cuánto resume la agregación (de ~250 datos al año a 12).

In [ ]:
# 1) El cierre mensual debe ser el Close del último día hábil de cada mes
ultimo_dia = datos["Close"].groupby([datos.index.year, datos.index.month]).tail(1)
chequeo_cierre = np.allclose(ultimo_dia.values, mensual["Cierre_mensual"].values)

# 2) Rendimiento recalculado manualmente
manual = (mensual["Cierre_mensual"] / mensual["Cierre_mensual"].shift(1) - 1) * 100
chequeo_rend = np.allclose(manual.dropna().values, mensual["Rendimiento_mensual_pct"].dropna().values)

print("¿Cierre mensual = último día hábil del mes? ->", chequeo_cierre)
print("¿Rendimiento mensual recalculado coincide?  ->", chequeo_rend)
print(f"\nObservaciones diarias: {len(datos)}  ->  observaciones mensuales: {len(mensual)}")
print(f"Reducción de datos: {100 * (1 - len(mensual) / len(datos)):.1f}%")
print(f"Meses con rendimiento calculable: {mensual['Rendimiento_mensual_pct'].notna().sum()}")

## 3. Graficar precios y rendimientos mensuales

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12,8), sharex=True)
ax[0].plot(mensual.index, mensual["Cierre_mensual"])
ax[0].set_title("AAPL: último precio disponible de cada mes")
ax[0].set_ylabel("Precio (USD)")
ax[0].grid(True, alpha=0.3)
ax[1].bar(mensual.index, mensual["Rendimiento_mensual_pct"], width=20)
ax[1].axhline(0, linewidth=1)
ax[1].set_title("Rendimientos mensuales")
ax[1].set_ylabel("Rendimiento (%)")
ax[1].set_xlabel("Fecha")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Revisión del gráfico (qué muestra cada panel):**
- *Panel superior:* el nivel del precio al cierre de cada mes (USD). Permite ver la **tendencia** de largo plazo y los grandes tramos alcistas o bajistas. El eje Y está en dólares y el eje X en fechas mensuales.
- *Panel inferior:* el cambio porcentual de un cierre al siguiente (%). Muestra la **magnitud y el signo** de cada mes y deja ver que los rendimientos fluctúan alrededor de cero sin un patrón visual evidente. Un gráfico de barras de rendimientos no sirve por sí solo para afirmar estacionalidad: hay que agrupar por mes calendario (sección 4).

### 3.1 ¿Qué cambia entre la frecuencia diaria y la mensual?
Comparamos la serie diaria con la mensual en nivel y en dispersión. Para poner la volatilidad diaria y mensual en la misma escala, anualizamos ambas (√252 para diaria, √12 para mensual). Si fueran consistentes, las dos cifras deberían ser parecidas; las diferencias muestran lo que se pierde o se distorsiona al resumir.

In [ ]:
rend_diario = datos["Close"].pct_change().dropna() * 100
rend_mensual = mensual["Rendimiento_mensual_pct"].dropna()

resumen = pd.DataFrame({
    "Observaciones":              [len(rend_diario), len(rend_mensual)],
    "Rend. promedio (%)":         [rend_diario.mean(), rend_mensual.mean()],
    "Desv. estándar (%)":         [rend_diario.std(), rend_mensual.std()],
    "Mínimo (%)":                 [rend_diario.min(), rend_mensual.min()],
    "Máximo (%)":                 [rend_diario.max(), rend_mensual.max()],
    "Volatilidad anualizada (%)": [rend_diario.std() * np.sqrt(252), rend_mensual.std() * np.sqrt(12)],
}, index=["Diaria", "Mensual"]).round(2)
display(resumen)

# Mayor movimiento diario dentro de cada mes vs. el rendimiento que "ve" la serie mensual
rango_intra = datos["Close"].resample("ME").agg(["min", "max"])
rango_intra["Amplitud_intramensual_pct"] = (rango_intra["max"] / rango_intra["min"] - 1) * 100
print("Amplitud promedio entre el mínimo y el máximo de cierre dentro de un mes:",
      round(rango_intra["Amplitud_intramensual_pct"].mean(), 2), "%")
print("Rendimiento mensual absoluto promedio (lo que ve la serie mensual):",
      round(rend_mensual.abs().mean(), 2), "%")

## 4. Exploración de meses del año
El promedio de rendimientos por mes calendario es solo una exploración inicial. Una diferencia entre promedios no demuestra que exista un efecto estacional estable ni garantiza que se repita.

In [ ]:
temp = datos[["Close"]].copy()
temp["Rendimiento_pct"] = temp["Close"].pct_change() * 100
temp["Mes"] = temp.index.month
promedio_mes = temp.groupby("Mes")["Rendimiento_pct"].agg(["mean", "median", "count"])
promedio_mes.index = ["Enero","Febrero","Marzo","Abril","Mayo","Junio","Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]
display(promedio_mes.round(3))
promedio_mes["mean"].plot(kind="bar", figsize=(11,4), title="Rendimiento diario promedio por mes calendario")
plt.xlabel("Mes")
plt.ylabel("Rendimiento diario promedio (%)")
plt.axhline(0, linewidth=1)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

**Cómo leerlo:** `mean` es el rendimiento *diario* promedio de los días de ese mes calendario (todos los años juntos); `median` es menos sensible a días extremos; `count` es el número de días usados. Si la media y la mediana difieren mucho, unos pocos días extremos están moviendo el promedio. Un valor alto en un mes **no** significa que se repita cada año: eso se evalúa en la sección 5.

## 5. Evidencia adicional: ¿el patrón se repite o es casual?
Una diferencia entre promedios es solo el primer paso. A continuación aplicamos los criterios de la sección 1: consistencia entre años, dispersión, prueba formal y descomposición.

### 5.1 Rendimiento mensual por mes calendario (con dispersión y consistencia)

In [ ]:
nombres = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]

m = mensual["Rendimiento_mensual_pct"].dropna().to_frame("rend")
m["Mes"] = m.index.month
m["Año"] = m.index.year

tabla = m.groupby("Mes")["rend"].agg(
    Promedio="mean", Mediana="median", Desv_est="std", Observaciones="count"
)
tabla["% de años con rendimiento > 0"] = m.groupby("Mes")["rend"].apply(lambda s: (s > 0).mean() * 100)
tabla.index = nombres
display(tabla.round(2))

### 5.2 Mapa de calor año × mes
Permite ver de un vistazo si un mismo mes tiene el mismo signo en los distintos años. Si hubiera estacionalidad estable, veríamos **columnas de color consistente**.

In [ ]:
pivote = m.pivot(index="Año", columns="Mes", values="rend")
pivote.columns = [nombres[i-1] for i in pivote.columns]

fig, ax = plt.subplots(figsize=(12, 4.5))
lim = np.nanmax(np.abs(pivote.values))
im = ax.imshow(pivote.values, cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(pivote.columns))); ax.set_xticklabels(pivote.columns)
ax.set_yticks(range(len(pivote.index)));   ax.set_yticklabels(pivote.index)
for i in range(pivote.shape[0]):
    for j in range(pivote.shape[1]):
        v = pivote.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8)
ax.set_title("AAPL: rendimiento mensual (%) por año y mes calendario")
ax.set_xlabel("Mes calendario")
ax.set_ylabel("Año")
fig.colorbar(im, ax=ax, label="Rendimiento mensual (%)")
plt.tight_layout()
plt.show()

### 5.3 Dispersión dentro de cada mes (diagrama de cajas)
Si la variación **dentro** de cada mes es grande frente a la diferencia **entre** meses, el patrón no es distinguible del ruido.

In [ ]:
grupos = [m.loc[m["Mes"] == k, "rend"].values for k in range(1, 13)]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.boxplot(grupos, showmeans=True)
ax.set_xticklabels(nombres)
ax.axhline(0, linewidth=1)
ax.set_title("AAPL: distribución del rendimiento mensual por mes calendario")
ax.set_xlabel("Mes calendario")
ax.set_ylabel("Rendimiento mensual (%)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 Prueba estadística formal
Usamos **Kruskal-Wallis**, una prueba no paramétrica que contrasta si los 12 meses provienen de la misma distribución (no exige normalidad, algo razonable con rendimientos financieros). Además ajustamos un ANOVA como contraste complementario.

- **H₀:** los rendimientos mensuales tienen la misma distribución en los 12 meses (no hay efecto del mes).
- **H₁:** al menos un mes difiere.

Limitación a tener presente: con ~6 años hay apenas 5 o 6 observaciones por mes calendario, así que la potencia de la prueba es baja. **No rechazar H₀ no prueba que no exista estacionalidad**; solo indica que estos datos no bastan para sostenerla.

In [ ]:
from scipy import stats

h, p_kw = stats.kruskal(*grupos)
f, p_an = stats.f_oneway(*grupos)

print(f"Kruskal-Wallis: H = {h:.3f}, p-valor = {p_kw:.4f}")
print(f"ANOVA         : F = {f:.3f}, p-valor = {p_an:.4f}")
print(f"Observaciones por mes calendario: mín = {tabla['Observaciones'].min()}, máx = {tabla['Observaciones'].max()}")

# Con 12 meses comparados a la vez, algunos van a parecer 'extremos' solo por azar.
# Contraste simple: ¿cuántos meses tienen promedio distinto de 0 con p < 0.05 (t de una muestra)?
pvals = [stats.ttest_1samp(g, 0).pvalue for g in grupos]
n_sig = sum(pv < 0.05 for pv in pvals)
print(f"\nMeses con promedio distinto de 0 (p < 0.05, sin corregir): {n_sig} de 12")
print(f"Falsos positivos esperados solo por azar con 12 pruebas al 5%: {12 * 0.05:.1f}")
print(f"Con corrección de Bonferroni (umbral {0.05/12:.4f}): {sum(pv < 0.05/12 for pv in pvals)} de 12")

### 5.5 Descomposición tendencia + estacionalidad + residuo
Descomponemos el **logaritmo** del precio mensual (para que las variaciones sean proporcionales) con periodo 12 usando un modelo aditivo. Medimos la **fuerza de la estacionalidad** como en Hyndman y Athanasopoulos: $F_S = \max\left(0,\ 1 - \frac{Var(R_t)}{Var(S_t + R_t)}\right)$, donde 0 indica ausencia de estacionalidad y 1 una estacionalidad muy fuerte.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

log_precio = np.log(mensual["Cierre_mensual"])
desc = seasonal_decompose(log_precio, model="additive", period=12)

fig = desc.plot()
fig.set_size_inches(12, 8)
fig.suptitle("AAPL: descomposición del log del precio mensual (periodo = 12)", y=1.02)
plt.tight_layout()
plt.show()

sr = (desc.seasonal + desc.resid).dropna()
fuerza_estacional = max(0, 1 - desc.resid.dropna().var() / sr.var())

tr = (desc.trend + desc.resid).dropna()
fuerza_tendencia = max(0, 1 - desc.resid.dropna().var() / tr.var())

print(f"Fuerza de la tendencia     : {fuerza_tendencia:.2f}")
print(f"Fuerza de la estacionalidad: {fuerza_estacional:.2f}")
print(f"Amplitud del componente estacional (log): {desc.seasonal.max() - desc.seasonal.min():.3f}")

### 5.6 Prueba de estabilidad entre períodos
Una verificación sencilla de "repetición": dividimos la muestra en dos mitades (2020–2022 y 2023–2025), calculamos el rendimiento promedio por mes en cada una y medimos la correlación entre ambos perfiles. Si hubiera un patrón estacional estable, los meses buenos de la primera mitad tenderían a ser buenos en la segunda.

In [ ]:
mitad_1 = m[m["Año"] <= 2022].groupby("Mes")["rend"].mean()
mitad_2 = m[m["Año"] >= 2023].groupby("Mes")["rend"].mean()

comp = pd.DataFrame({"2020-2022": mitad_1, "2023-2025": mitad_2})
comp.index = nombres
display(comp.round(2))

rho, p_rho = stats.spearmanr(comp["2020-2022"], comp["2023-2025"])
print(f"Correlación de Spearman entre los dos perfiles mensuales: {rho:.2f} (p-valor = {p_rho:.3f})")

ax = comp.plot(kind="bar", figsize=(11, 4), title="Rendimiento mensual promedio: primera vs. segunda mitad")
ax.axhline(0, linewidth=1)
ax.set_xlabel("Mes calendario")
ax.set_ylabel("Rendimiento mensual promedio (%)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 5.7 Resumen automático de la evidencia
Esta celda lee los resultados anteriores y redacta un resumen que se actualiza con los datos descargados, para que el texto nunca contradiga las cifras.

In [ ]:
mejor = tabla["Promedio"].idxmax(); peor = tabla["Promedio"].idxmin()
umbral = 0.05

print("RESUMEN DE EVIDENCIA")
print("-" * 60)
print(f"Mes con mayor rendimiento mensual promedio : {mejor} ({tabla.loc[mejor,'Promedio']:.2f}%)")
print(f"Mes con menor rendimiento mensual promedio : {peor} ({tabla.loc[peor,'Promedio']:.2f}%)")
print(f"Desv. estándar típica dentro de un mes     : {tabla['Desv_est'].mean():.2f}%")
print(f"Diferencia entre el mejor y el peor mes    : {tabla['Promedio'].max() - tabla['Promedio'].min():.2f} pp")
print(f"Observaciones por mes calendario           : {tabla['Observaciones'].min()}-{tabla['Observaciones'].max()}")
print(f"Kruskal-Wallis p-valor                     : {p_kw:.4f}")
print(f"Fuerza de estacionalidad (0-1)             : {fuerza_estacional:.2f}")
print(f"Correlación entre mitades (Spearman)       : {rho:.2f}")
print("-" * 60)

criterios = {
    "Prueba formal rechaza que los meses sean iguales (p < 0.05)": p_kw < umbral,
    "Estacionalidad con peso relevante (F_S >= 0.64 según Hyndman)": fuerza_estacional >= 0.64,
    "Perfil mensual se mantiene entre mitades (rho > 0 y p < 0.05)": (rho > 0) and (p_rho < umbral),
}
for k, v in criterios.items():
    print(("[CUMPLE]    " if v else "[NO CUMPLE] ") + k)

print()
if all(criterios.values()):
    print("Veredicto: hay indicios consistentes de estacionalidad, que igual requerirían validación fuera de muestra.")
elif any(criterios.values()):
    print("Veredicto: evidencia parcial; no alcanza para afirmar estacionalidad.")
else:
    print("Veredicto: los datos no sostienen estacionalidad; las diferencias entre meses son compatibles con fluctuación casual.")

## 6. Interpretación

**1. ¿Qué detalle se hace más visible al agregar por mes?**
La **tendencia de largo plazo** y la magnitud de los movimientos entre cierres. El ruido diario desaparece y se ve con claridad la trayectoria del precio y los meses de rendimientos fuertes (positivos o negativos). Además, la frecuencia mensual permite agrupar por mes calendario, lo que es la base para preguntarse por estacionalidad. La tabla de la sección 3.1 cuantifica la diferencia entre ambas frecuencias.

**2. ¿Qué información diaria se pierde?**
Se pierde todo lo que ocurre **dentro** del mes: la volatilidad diaria, los picos y caídas intra-mes, los *gaps* por resultados y la trayectoria entre un cierre y el siguiente (dos meses con el mismo rendimiento pueden haber tenido recorridos muy distintos; la amplitud intramensual calculada en 3.1 lo evidencia). También se pierde la posibilidad de analizar efectos de corto plazo (día de la semana, eventos puntuales) y se reduce el tamaño de muestra de ~1.500 a 72 observaciones, lo que debilita cualquier inferencia estadística. Por último, el resultado depende de la regla de agregación: usar el cierre del mes o el promedio del mes da series distintas.

**3. ¿Qué diferencias observan entre meses y por qué no bastan para probar estacionalidad?**
Las tablas y gráficos de las secciones 4 y 5 muestran que unos meses tienen promedios mayores que otros (la celda 5.7 identifica cuáles con los datos descargados). Aun así, esas diferencias **no bastan** porque:
- Hay muy pocas observaciones por mes (5 o 6, una por año), así que un solo año extraordinario mueve el promedio.
- La dispersión dentro de cada mes es grande frente a la diferencia entre meses (diagrama de cajas, 5.3).
- Se comparan 12 meses a la vez: por puro azar es esperable que alguno parezca "especial" (problema de comparaciones múltiples, 5.4).
- El período 2020–2025 incluye eventos únicos (caída por COVID-19 en 2020, rebote posterior, ciclo de alzas de tasas en 2022), que no son calendario y contaminan los promedios por mes.
- Un promedio alto no implica que el signo se repita cada año (mapa de calor, 5.2, y columna "% de años con rendimiento > 0").
- La tendencia alcista de largo plazo eleva el promedio de todos los meses y puede confundirse con efecto estacional si no se separa.

**4. ¿Qué análisis adicional realizarían para evaluar si un patrón se repite?**
Ya incluimos: consistencia por año (mapa de calor), dispersión (cajas), prueba formal (Kruskal-Wallis/ANOVA con corrección por comparaciones múltiples), descomposición con fuerza estacional y estabilidad entre mitades. Como siguiente paso haríamos:
- Ampliar la muestra a 20–30 años para tener más observaciones por mes.
- **Validación fuera de muestra** (*walk-forward*): estimar el patrón con unos años y comprobarlo en los siguientes.
- Trabajar con **rendimientos en exceso** frente a un índice (por ejemplo, S&P 500) para aislar el efecto del mercado.
- Comparar con otros activos para ver si el patrón es propio de AAPL o general.
- Incluir regresión con variables dummy mensuales y errores robustos, y considerar costos de transacción si se quisiera operar el patrón.
- Buscar un mecanismo económico que justifique el patrón (por ejemplo, calendario de lanzamientos de producto o de resultados trimestrales).

## 7. Conclusión del equipo

Al pasar de datos diarios a mensuales cambia la **resolución** de la serie: se gana claridad sobre la tendencia y la posibilidad de comparar meses calendario, y se pierde la dinámica intra-mes y buena parte del tamaño de muestra. La agregación no es neutra: usar el último cierre del mes no es lo mismo que promediar el mes.

Sobre la pregunta central, **ver diferencias entre los promedios de los meses no es evidencia de estacionalidad**. Para sostenerla se necesitaría que el patrón sea consistente entre años, que supere la variabilidad propia del activo, que pase una prueba formal que tenga en cuenta las comparaciones múltiples, que la estacionalidad pese en la descomposición y que se mantenga fuera de la muestra. El resumen automático de la sección 5.7 evalúa estos criterios con los datos descargados y emite el veredicto correspondiente; con apenas seis años de historia y solo 5 o 6 observaciones por mes, lo esperable es que la evidencia sea limitada, y en ese caso las diferencias deben tratarse como fluctuación casual y no como señal.

**¿Qué haría un analista con esta información?** No construiría una estrategia ni una proyección basada en un "efecto mes" con estos datos. Usaría el análisis como exploración, ampliaría la historia, validaría fuera de muestra y exigiría un mecanismo económico antes de dar crédito a cualquier patrón calendario. Mientras tanto, la tendencia y la volatilidad serían los elementos más informativos de la serie.


## 8. Fuentes y participación

**Referencias bibliográficas**

Banco de España. (2024, septiembre). Serie temporal. https://www.bde.es/webbe/es/estadisticas/recursos/glosario/conceptos/serie-temporal.html 

Departamento Administrativo Nacional de Estadística. (2016, 16 de septiembre). ¿Qué tipo de datos usa en su vida cotidiana?. https://www.dane.gov.co/index.php/actualidad-dane/3886-que-tipo-de-datos-usa-en-su-vida-cotidiana-3 

Instituto Nacional de Estadística. (s. f.). Estándar del INE para la corrección de efectos estacionales y efectos de calendario en las series coyunturales. https://ine.es/clasifi/estandar_efectos_estacionales.pdf 


**Aportes de integrantes** (completar con los nombres del equipo):

| Rol | Integrante | Aporte |
|---|---|---|
| Responsable del código | [Sebastián Giraldo Castaño] | Subió el cuaderno a Colab, lo compartió con el equipo y ejecutó las celdas |
| Investigación conceptual | [Isabel Cristina Restrepo Cuervo] | Redactó las definiciones y citó las fuentes |
| Revisión de cálculos | [Hernán Alonso Gómez Meza] | Verificó fechas, cifras y unidades (sección 2.1) |
| Revisión de gráficos | [Hernán Alonso Gómez Meza] | Comprobó títulos, ejes y unidades de cada gráfico |
| Conclusión | [Isabel Cristina Restrepo Cuervo] | Redactó la interpretación final |
